# PatchVision — training notebook (GvT + bounding-box attention)

Runs the model code in `patchvision_gvt.py` (same folder) and records every step and epoch.

**Variants** (set `VARIANT` below)
| `VARIANT` | what it is |
|---|---|
| `vit` | plain ViT control (linear patch embedding + CLS token) |
| `gvt` | Graph-based ViT with sparsity (Sci. Rep. 2025) |
| `gvt_bbox` | `gvt` **+ our add-on**: bounding-box attention loss (boxes used in training only) |

**Training recipe** — AdamW (wd 0.05) · peak LR 5e-4 · 10-epoch linear warm-up then cosine decay to 1e-6 (per step) ·
batch 64 · **300 epochs** (paper's setting for transformers) · label smoothing 0.1 · mixup/CutMix from `preprocess_config.json` · grad-clip 1.0.

**How to run**
1. Put `patchvision_gvt.py` next to this notebook, and run it from the folder that contains `data/` (the same place your preprocessing notebooks ran).
2. First set `SMOKE_TEST = True` and *Run all* (≈ 2–5 min): it trains 2 epochs on synthetic images and checks the whole pipeline.
3. Set `SMOKE_TEST = False` and *Run all* for the real run. Everything is saved in `runs/<variant>_seed<seed>/`.
4. If the session dies, just *Run all* again: `RESUME = True` continues from `last.pt`.

In [ ]:
import json, math, os, random, sys, time, warnings, dataclasses
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

import patchvision_gvt as pv          # <- patchvision_gvt.py must sit next to this notebook
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())

## 1 · Settings

In [ ]:
# ================================ USER SETTINGS ================================
SMOKE_TEST       = False        # True = 2-epoch dry run on synthetic data (checks the whole pipeline)
VARIANT          = "gvt_bbox"   # "vit" | "gvt" | "gvt_bbox"
SEEDS            = [42]         # the paper averages 5 runs -> [42, 43, 44, 45, 46]
EPOCHS           = 300          # paper: 300 epochs for transformers
AUG_TIER         = "strong"     # "none" | "basic" | "strong" | "auto"   (from preprocess_config.json)
USE_MIXUP_CUTMIX = True
BEST_METRIC      = "acc1"       # checkpoint selection on validation: "acc1" | "macro_f1" | "loss"(lower is better)
STEP_PRINT_EVERY = 1            # 1 = print EVERY step (very verbose). Raise to 5/10 if the notebook gets slow.
                                #   (the log file always contains every step)
NUM_WORKERS      = 2            # set to 0 if you hit the shared-memory (shm) crash you saw before
RESUME           = True
ROOT             = Path("data") # same as in the preprocessing notebooks (train.csv, val.csv, test.csv, *.json)
RUN_DIR          = Path("runs")

MODEL_OVERRIDES  = {}           # e.g. {"embed_dim": 384, "depth": 6, "heads": 6}   (defaults: dim 256, 8 blocks, 8 heads)
TRAIN_OVERRIDES  = {}           # e.g. {"attn_loss_weight": 0.2, "lr": 3e-4}
# ===============================================================================

## 2 · Data, configs and the model/optimizer setup

In [ ]:
DEVICE = pv.get_device()
if SMOKE_TEST:
    pv.make_synthetic_cub("smoke_data", n_classes=8, n_train=12, n_val=2, n_test=4)
    ROOT, DATA_BASE, RUN_DIR = Path("smoke_data/data"), Path("smoke_data"), Path("runs_smoke")
    cfg_json = {"img_size": 224, "batch_size": 16, "mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225],
                "mixup_cutmix": {"mixup_alpha": 0.2, "cutmix_alpha": 1.0, "p_cutmix": 0.5}}
    EPOCHS, WARMUP_EPOCHS = 2, 1
    print(">>> SMOKE TEST: synthetic data, 2 epochs <<<")
else:
    DATA_BASE = Path(".")      # the csv 'filepath' column already starts with data/CUB_200_2011/...
    cfg_json = json.loads((ROOT / "preprocess_config.json").read_text())
    WARMUP_EPOCHS = 10

IMG_SIZE, BATCH_SIZE = cfg_json["img_size"], cfg_json["batch_size"]
MIXCFG = cfg_json["mixup_cutmix"]
if SMOKE_TEST:
    MEAN, STD = cfg_json["mean"], cfg_json["std"]
else:
    _stats = json.loads((ROOT / "norm_stats.json").read_text())          # normalization comes from norm_stats.json
    MEAN, STD = _stats["mean"], _stats["std"]
    assert np.allclose(MEAN, cfg_json["mean"]) and np.allclose(STD, cfg_json["std"]), \
        "norm_stats.json and preprocess_config.json disagree on mean/std"
print("normalization (norm_stats.json): mean", MEAN, "std", STD)
NUM_CLASSES = pd.read_csv(ROOT / "train.csv")["label"].nunique()

BBOX_FILE = ROOT / "CUB_200_2011" / "bounding_boxes.txt"
BBOXES = pv.load_bboxes(BBOX_FILE)
if VARIANT == "gvt_bbox" and BBOXES is None:
    raise FileNotFoundError(f"{BBOX_FILE} not found - the gvt_bbox variant needs CUB's bounding_boxes.txt")
print(f"bounding boxes loaded: {0 if BBOXES is None else len(BBOXES)} | classes: {NUM_CLASSES} | device: {DEVICE}")

MCFG = pv.ModelConfig(variant=VARIANT, img_size=IMG_SIZE, num_classes=NUM_CLASSES, **MODEL_OVERRIDES)
TC = pv.TrainConfig(epochs=EPOCHS, batch_size=BATCH_SIZE, warmup_epochs=WARMUP_EPOCHS,
                    use_mixup_cutmix=USE_MIXUP_CUTMIX, **TRAIN_OVERRIDES)
GRID = IMG_SIZE // MCFG.patch_size
USE_AMP = TC.amp and DEVICE.type == "cuda"
AMP_DTYPE = torch.bfloat16 if (DEVICE.type == "cuda" and torch.cuda.get_device_capability()[0] >= 8) else torch.float16
print("model config :", MCFG)
print("train config :", TC)
print(f"AMP: {USE_AMP} ({AMP_DTYPE})")

def make_loaders(seed):
    tr_tf = pv.build_transforms(AUG_TIER, IMG_SIZE, MEAN, STD, train=True)
    ev_tf = pv.build_transforms("none", IMG_SIZE, MEAN, STD, train=False)
    ds = {"train": pv.CUBDataset(ROOT / "train.csv", tr_tf, BBOXES, DATA_BASE),
          "val":   pv.CUBDataset(ROOT / "val.csv",   ev_tf, BBOXES, DATA_BASE),
          "test":  pv.CUBDataset(ROOT / "test.csv",  ev_tf, BBOXES, DATA_BASE)}
    g = torch.Generator(); g.manual_seed(seed)
    def winit(_):
        s = torch.initial_seed() % 2**32; np.random.seed(s); random.seed(s)
    kw = dict(num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda", worker_init_fn=winit,
              persistent_workers=NUM_WORKERS > 0)
    dl = {"train": DataLoader(ds["train"], BATCH_SIZE, shuffle=True, drop_last=True, generator=g, **kw),
          "val":   DataLoader(ds["val"],   BATCH_SIZE, shuffle=False, **kw),
          "test":  DataLoader(ds["test"],  BATCH_SIZE, shuffle=False, **kw)}
    return ds, dl

ds, dl = make_loaders(SEEDS[0])
print({k: len(v) for k, v in ds.items()}, "| expected splits from config:", cfg_json.get("splits"))
print(f"steps per epoch: {len(dl['train'])}  ->  total optimizer steps: {len(dl['train']) * EPOCHS:,}")

### Sanity check — augmented images with their bird-box (green). The box must stay on the bird.

In [ ]:
RUN_DIR.mkdir(exist_ok=True, parents=True)
mean_t, std_t = torch.tensor(MEAN)[:, None, None], torch.tensor(STD)[:, None, None]
fig, axes = plt.subplots(2, 6, figsize=(18, 6.5))
tr_ds = ds["train"]
for ax, i in zip(axes.ravel(), range(12)):
    img, m, y, iid = tr_ds[i]
    ax.imshow((img * std_t + mean_t).clamp(0, 1).permute(1, 2, 0).numpy())
    if m.max() > 0.5: ax.contour(m[0].numpy(), levels=[0.5], colors="lime", linewidths=2)
    ax.set_title(f"{tr_ds.class_names[y][:18]}", fontsize=9); ax.axis("off")
plt.suptitle(f"augmentation tier = '{AUG_TIER}'  (green = bounding-box mask after the same augmentation)")
plt.tight_layout(); plt.savefig(RUN_DIR / "augmentation_check.png", dpi=110); plt.show()

### Dry run — model summary, FLOPs, forward / backward on one REAL batch, gradient check

In [ ]:
seed_dry = SEEDS[0]
pv.seed_everything(seed_dry)
model = pv.build_model(MCFG).to(DEVICE)
total_p, train_p = pv.count_parameters(model)
print(f"variant {VARIANT}: {total_p/1e6:.2f} M parameters ({train_p/1e6:.2f} M trainable)")
eff = pv.measure_efficiency(model, DEVICE, IMG_SIZE, batch_sizes=(1,), iters=3)
print("params / FLOPs / latency (batch 1):", {k: round(v, 3) for k, v in eff.items()})

x, m, y, _ = next(iter(dl["train"]))
x, m, y = x.to(DEVICE), m.to(DEVICE), y.to(DEVICE)
crit = pv.TrainingLoss(VARIANT == "gvt_bbox", TC.attn_loss_weight)
model.train()
logits, aux = model(x)
out = crit(logits.float(), pv.smooth_one_hot(y, NUM_CLASSES, TC.label_smoothing), aux, pv.box_to_patch_mask(m, GRID))
out["total"].backward()
missing = [n for n, p in model.named_parameters() if p.grad is None or not torch.isfinite(p.grad).all()]
print(f"logits {tuple(logits.shape)} | attention maps used by the bbox loss: {len(aux['attn_maps'])} x {tuple(aux['attn_maps'][0].shape)}")
print(f"initial loss {out['total'].item():.4f} (ce {out['ce'].item():.4f}, attn {out['attn'].item():.4f}) | "
      f"chance-level CE = ln({NUM_CLASSES}) = {math.log(NUM_CLASSES):.4f}")
print("parameters without a finite gradient:", missing if missing else "none  [OK]")
assert not missing
del model, crit, out, logits, aux, x, m, y
import gc; del dl; gc.collect()          # release the dry-run data workers

## 3 · Training and evaluation functions (verbose)

In [ ]:
def fmt(d, k, spec=".2f"):
    return format(d[k], spec) if k in d else "n/a"

class Logger:
    """print() + append every line to a log file (the file always has everything)."""
    def __init__(self, path): self.path = Path(path); self.path.parent.mkdir(parents=True, exist_ok=True)
    def __call__(self, msg, show=True):
        if show: print(msg)
        with open(self.path, "a") as f: f.write(msg + "\n")

def train_one_epoch(model, loader, criterion, optimizer, scheduler, scaler, mixer, epoch, global_step, run_start, log, step_rows):
    model.train()
    tracker = pv.MetricTracker(NUM_CLASSES)
    n_steps, t_epoch = len(loader), time.time()
    sums = {"total": 0.0, "ce": 0.0, "attn": 0.0}
    gnorm_sum, data_time, n_img, skipped = 0.0, 0.0, 0, 0
    lr_first = optimizer.param_groups[0]["lr"]
    if DEVICE.type == "cuda": torch.cuda.reset_peak_memory_stats()
    t_prev = time.time()
    for step, (x, m, y, _) in enumerate(loader, 1):
        t_data = time.time() - t_prev
        t0 = time.time()
        x, m, y = x.to(DEVICE, non_blocking=True), m.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        if mixer is not None:
            x, soft, m, mixinfo = mixer(x, y, m)
        else:
            soft, mixinfo = pv.smooth_one_hot(y, NUM_CLASSES, TC.label_smoothing), {"type": "none", "lam": 1.0}
        hard = soft.argmax(1)                                   # dominant label (used for train accuracy)
        pmask = pv.box_to_patch_mask(m, GRID)

        lr_now = optimizer.param_groups[0]["lr"]
        with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
            logits, aux = model(x)
        out = criterion(logits.float(), soft, aux, pmask)
        if not torch.isfinite(out["total"]):
            skipped += 1
            log(f"  !! step {step}: non-finite loss ({out['total'].item()}) -> step skipped", True)
            optimizer.zero_grad(set_to_none=True); t_prev = time.time(); continue
        optimizer.zero_grad(set_to_none=True)
        if scaler is not None:
            scaler.scale(out["total"]).backward(); scaler.unscale_(optimizer)
            gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), TC.grad_clip).item()
            scaler.step(optimizer); scaler.update()
        else:
            out["total"].backward()
            gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), TC.grad_clip).item()
            optimizer.step()
        scheduler.step()
        global_step += 1

        b = y.shape[0]
        step_time = time.time() - t0
        pred5 = logits.detach().topk(min(5, NUM_CLASSES), 1).indices
        a1 = (pred5[:, 0] == hard).float().mean().item() * 100
        a5 = (pred5 == hard[:, None]).any(1).float().mean().item() * 100
        valid = out["valid"]
        mass = (out["mass"] * valid).sum().item() / max(1.0, valid.sum().item())
        area = ((pmask.sum(1) / pmask.shape[1]) * valid).sum().item() / max(1.0, valid.sum().item())
        gm, gs = aux["gate_mean"].item(), aux["gate_sparsity"].item()
        tot, ce, at = out["total"].item(), out["ce"].item(), out["attn"].item()
        tracker.update(logits.detach().float(), hard, tot, mass=out["mass"], valid=valid,
                       area=pmask.sum(1) / pmask.shape[1], gate=(gm, gs))
        sums["total"] += tot; sums["ce"] += ce; sums["attn"] += at
        gnorm_sum += gnorm; data_time += t_data; n_img += b
        ips = b / max(1e-9, step_time + t_data)
        elapsed = time.time() - t_epoch
        eta_epoch = elapsed / step * (n_steps - step)
        mem = torch.cuda.max_memory_allocated() / 1e9 if DEVICE.type == "cuda" else 0.0
        w = TC.attn_loss_weight if VARIANT == "gvt_bbox" else 0.0
        line = (f"[E{epoch:03d}/{EPOCHS} | S{step:03d}/{n_steps} | g{global_step:06d}] "
                f"loss {tot:.4f} (ce {ce:.4f} + {w:g}*attn {at:.4f}) | acc1 {a1:5.1f}% acc5 {a5:5.1f}% | "
                f"in-box {mass:.3f} (box area {area:.3f}) | {mixinfo['type']}(lam={mixinfo['lam']:.2f}) | "
                f"lr {lr_now:.3e} | gnorm {gnorm:6.3f} | gate mean {gm:.2f} sparse {gs:.2f} | "
                f"{ips:5.0f} img/s (data wait {t_data*1000:4.0f} ms) | mem {mem:.2f} GB | "
                f"step {step_time:.2f}s | epoch ETA {pv.fmt_time(eta_epoch)}")
        log(line, show=(step % STEP_PRINT_EVERY == 0 or step == n_steps))
        step_rows.append({"epoch": epoch, "step": step, "global_step": global_step, "loss": tot, "ce": ce, "attn": at,
                          "acc1": a1, "acc5": a5, "attn_in_box": mass, "box_area": area, "lr": lr_now, "gnorm": gnorm,
                          "gate_mean": gm, "gate_sparsity": gs, "img_per_s": ips, "data_wait_ms": t_data * 1000,
                          "mem_gb": mem, "mix": mixinfo["type"], "lam": mixinfo["lam"]})
        t_prev = time.time()
    res = tracker.compute()
    res.update({"ce": sums["ce"] / n_steps, "attn": sums["attn"] / n_steps, "gnorm": gnorm_sum / n_steps,
                "lr_first": lr_first, "lr_last": optimizer.param_groups[0]["lr"], "time_s": time.time() - t_epoch,
                "data_wait_frac": data_time / max(1e-9, time.time() - t_epoch), "skipped": skipped})
    return res, global_step

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    tracker, attn_loss = pv.MetricTracker(NUM_CLASSES), pv.BBoxAttentionLoss()
    t0 = time.time()
    for x, m, y, _ in loader:
        x, m, y = x.to(DEVICE), m.to(DEVICE), y.to(DEVICE)
        pmask = pv.box_to_patch_mask(m, GRID)
        with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
            logits, aux = model(x)
        logits = logits.float()
        _, mass, valid = attn_loss(aux["attn_maps"], pmask)
        tracker.update(logits, y, F.cross_entropy(logits, y).item(), mass=mass, valid=valid,
                       area=pmask.sum(1) / pmask.shape[1], gate=(aux["gate_mean"].item(), aux["gate_sparsity"].item()))
    res = tracker.compute(); res["time_s"] = time.time() - t0
    return res, tracker

def is_better(new, best):
    if best is None: return True
    return new < best if BEST_METRIC == "loss" else new > best

## 4 · One full experiment (train → keep best model → test)

In [ ]:
def run_experiment(seed):
    run_name = f"{VARIANT}_seed{seed}"
    rd = RUN_DIR / run_name; rd.mkdir(parents=True, exist_ok=True)
    log = Logger(rd / "train_log.txt")
    pv.seed_everything(seed)
    ds_, dl_ = make_loaders(seed)
    model = pv.build_model(MCFG).to(DEVICE)
    opt = pv.build_optimizer(model, TC)
    sched = pv.build_scheduler(opt, TC, len(dl_["train"]))
    scaler = torch.amp.GradScaler("cuda") if (USE_AMP and AMP_DTYPE == torch.float16) else None
    criterion = pv.TrainingLoss(VARIANT == "gvt_bbox", TC.attn_loss_weight)
    mixer = pv.MixupCutmix(NUM_CLASSES, MIXCFG["mixup_alpha"], MIXCFG["cutmix_alpha"], MIXCFG["p_cutmix"],
                           TC.label_smoothing) if TC.use_mixup_cutmix else None
    history, step_rows, best, best_epoch, start_epoch, global_step = [], [], None, 0, 1, 0

    last_path, best_path = rd / "last.pt", rd / "best_model.pt"
    if RESUME and last_path.exists():
        ck = pv.load_checkpoint(last_path, model, opt, sched, scaler, map_location=DEVICE)
        history, best, start_epoch = ck["history"], ck["best"], ck["epoch"] + 1
        best_epoch = ck["extra"].get("best_epoch", 0); global_step = ck["extra"].get("global_step", 0)
        log(f"### RESUMED from {last_path} at epoch {start_epoch} (best {BEST_METRIC} so far {best} @ epoch {best_epoch})")
    else:
        for f in ("train_log.txt", "steps.csv"):
            if (rd / f).exists(): (rd / f).unlink()
        log = Logger(rd / "train_log.txt")
        log("=" * 120)
        log(f"RUN {run_name} | variant {VARIANT} | seed {seed} | aug '{AUG_TIER}' | mixup/cutmix {TC.use_mixup_cutmix}")
        log(f"model {pv.count_parameters(model)[0]/1e6:.2f} M params | {MCFG}")
        log(f"train {TC}")
        log(f"steps/epoch {len(dl_['train'])} | total steps {len(dl_['train'])*EPOCHS:,} | device {DEVICE} | AMP {USE_AMP}")
        log("=" * 120)

    run_start = time.time(); epoch_times = []
    for epoch in range(start_epoch, EPOCHS + 1):
        log(f"\n{'-'*60} EPOCH {epoch:03d}/{EPOCHS} {'-'*60}")
        tr, global_step = train_one_epoch(model, dl_["train"], criterion, opt, sched, scaler, mixer, epoch,
                                          global_step, run_start, log, step_rows)
        va, _ = evaluate(model, dl_["val"])
        epoch_times.append(tr["time_s"] + va["time_s"])
        eta = np.mean(epoch_times[-10:]) * (EPOCHS - epoch)
        score = va[BEST_METRIC]
        improved = is_better(score, best)
        if improved:
            best, best_epoch = score, epoch
            pv.save_checkpoint(best_path, model, opt, sched, scaler, epoch, best, history, MCFG, TC,
                               {"val": va, "epoch": epoch, "seed": seed, "best_metric": BEST_METRIC})
        rec = {"epoch": epoch, **{f"train_{k}": v for k, v in tr.items()}, **{f"val_{k}": v for k, v in va.items()},
               "gap_acc1": tr["acc1"] - va["acc1"], "is_best": improved}
        history.append(rec)
        log("")
        log(f"  TRAIN  loss {tr['loss']:.4f} (ce {tr['ce']:.4f}, attn {tr['attn']:.4f}) | top-1 {tr['acc1']:.2f}% top-5 {tr['acc5']:.2f}% | "
            f"macro P/R/F1 {tr['macro_precision']:.2f}/{tr['macro_recall']:.2f}/{tr['macro_f1']:.2f} | weighted F1 {tr['weighted_f1']:.2f}   "
            f"(accuracy vs dominant label under mixup/cutmix: approximate)")
        log(f"  VAL    loss {va['loss']:.4f}                    | top-1 {va['acc1']:.2f}% top-5 {va['acc5']:.2f}% | "
            f"macro P/R/F1 {va['macro_precision']:.2f}/{va['macro_recall']:.2f}/{va['macro_f1']:.2f} | weighted F1 {va['weighted_f1']:.2f}   (n={va['n']})")
        log(f"  GAP    train-val top-1 {tr['acc1']-va['acc1']:+.2f} pts | val-loss minus train-ce {va['loss']-tr['ce']:+.4f}")
        log(f"  ATTN   share of attention inside the bird box: train {fmt(tr,'attn_in_box','.3f')} | val {fmt(va,'attn_in_box','.3f')}   "
            f"(box covers {fmt(va,'box_area_frac','.3f')} of the image -> chance level) | gate mean {fmt(va,'gate_mean','.3f')}, sparsity {fmt(va,'gate_sparsity','.3f')}")
        log(f"  OPT    lr {tr['lr_first']:.3e} -> {tr['lr_last']:.3e} | mean grad-norm {tr['gnorm']:.3f} | skipped steps {tr['skipped']}")
        log(f"  TIME   epoch {pv.fmt_time(tr['time_s'])} (train) + {pv.fmt_time(va['time_s'])} (val) | data-wait {tr['data_wait_frac']*100:.0f}% | "
            f"run elapsed {pv.fmt_time(time.time()-run_start)} | ETA {pv.fmt_time(eta)}")
        log(f"  BEST   val {BEST_METRIC} {best:.4f} @ epoch {best_epoch}" + ("   <== NEW BEST, model saved to " + str(best_path) if improved else ""))
        pd.DataFrame(history).to_csv(rd / "history.csv", index=False)
        if step_rows:
            pd.DataFrame(step_rows).to_csv(rd / "steps.csv", mode="a", index=False, header=not (rd / "steps.csv").exists())
            step_rows.clear()
        pv.save_checkpoint(last_path, model, opt, sched, scaler, epoch, best, history, MCFG, TC,
                           {"best_epoch": best_epoch, "global_step": global_step})

    # ------------------------------ final TEST on the best checkpoint ------------------------------
    log("\n" + "=" * 120 + f"\nTRAINING DONE. Loading best model (epoch {best_epoch}, val {BEST_METRIC} {best}) for the TEST set\n" + "=" * 120)
    pv.load_checkpoint(best_path, model, map_location=DEVICE)
    te, trk = evaluate(model, dl_["test"])
    eff = pv.measure_efficiency(model, DEVICE, IMG_SIZE, batch_sizes=(1, BATCH_SIZE), iters=20)
    log(f"TEST   loss {te['loss']:.4f} | top-1 {te['acc1']:.2f}% top-5 {te['acc5']:.2f}% | macro P/R/F1 "
        f"{te['macro_precision']:.2f}/{te['macro_recall']:.2f}/{te['macro_f1']:.2f} | weighted F1 {te['weighted_f1']:.2f} | n={te['n']}")
    log(f"TEST   attention inside bird box {fmt(te,'attn_in_box','.3f')} (chance {fmt(te,'box_area_frac','.3f')}) | gate sparsity {fmt(te,'gate_sparsity','.3f')}")
    log("EFFICIENCY " + " | ".join(f"{k} {v:.3f}" for k, v in eff.items()))
    report = pd.DataFrame(trk.per_class_report(ds_["test"].class_names)); report.to_csv(rd / "test_per_class.csv", index=False)
    conf = trk.conf.clone(); np.save(rd / "test_confusion_matrix.npy", conf.numpy())
    log("worst 10 classes (F1):\n" + report.sort_values("f1").head(10).to_string(index=False))
    log("best 10 classes (F1):\n" + report.sort_values("f1", ascending=False).head(10).to_string(index=False))
    off = conf.clone(); off.fill_diagonal_(0)
    top = torch.topk(off.flatten(), min(15, off.numel()))
    pairs = [(ds_["test"].class_names[i // NUM_CLASSES], ds_["test"].class_names[i % NUM_CLASSES], int(v))
             for v, i in zip(top.values, top.indices) if v > 0]
    log("most frequent confusions (true -> predicted : count):\n" + "\n".join(f"  {a} -> {b} : {n}" for a, b, n in pairs))
    json.dump({"run": run_name, "seed": seed, "variant": VARIANT, "best_epoch": best_epoch, "best_val_" + BEST_METRIC: best,
               "test": te, "efficiency": eff, "model_cfg": dataclasses.asdict(MCFG), "train_cfg": dataclasses.asdict(TC),
               "aug_tier": AUG_TIER}, open(rd / "test_metrics.json", "w"), indent=2, default=float)

    fig, ax = plt.subplots(figsize=(9, 8)); ax.imshow(conf.numpy(), cmap="viridis"); ax.set_xlabel("predicted"); ax.set_ylabel("true")
    ax.set_title(f"{run_name} test confusion matrix"); plt.tight_layout(); plt.savefig(rd / "test_confusion_matrix.png", dpi=120); plt.show()
    plot_curves(pd.DataFrame(history), rd, run_name)
    return {"seed": seed, "best_epoch": best_epoch, "run_dir": str(rd), **{f"test_{k}": v for k, v in te.items() if k in
            ("loss", "acc1", "acc5", "macro_precision", "macro_recall", "macro_f1", "weighted_f1", "attn_in_box")},
            "params_M": eff["params_M"], "GFLOPs": eff["GFLOPs_per_image"], "latency_ms_bs1": eff["latency_ms_bs1"]}

def plot_curves(h, rd, title):
    fig, ax = plt.subplots(2, 3, figsize=(17, 8)); e = h["epoch"]
    ax[0, 0].plot(e, h["train_loss"], label="train (total)"); ax[0, 0].plot(e, h["train_ce"], label="train ce", ls=":")
    ax[0, 0].plot(e, h["val_loss"], label="val ce"); ax[0, 0].set_title("loss"); ax[0, 0].legend()
    ax[0, 1].plot(e, h["train_acc1"], label="train"); ax[0, 1].plot(e, h["val_acc1"], label="val"); ax[0, 1].set_title("top-1 (%)"); ax[0, 1].legend()
    ax[0, 2].plot(e, h["val_macro_f1"], label="val macro F1"); ax[0, 2].plot(e, h["val_acc5"], label="val top-5"); ax[0, 2].set_title("val F1 / top-5"); ax[0, 2].legend()
    ax[1, 0].plot(e, h["train_lr_first"]); ax[1, 0].set_title("learning rate (start of epoch)")
    if "val_attn_in_box" in h:
        ax[1, 1].plot(e, h["val_attn_in_box"], label="attention inside box"); ax[1, 1].plot(e, h["val_box_area_frac"], label="box area (chance)", ls="--")
        ax[1, 1].set_title("val attention inside the bird box"); ax[1, 1].legend()
    ax[1, 2].plot(e, h["gap_acc1"]); ax[1, 2].axhline(0, c="k", lw=.5); ax[1, 2].set_title("train - val top-1 gap (overfitting)")
    for a in ax.ravel(): a.set_xlabel("epoch"); a.grid(alpha=.3)
    plt.suptitle(title); plt.tight_layout(); plt.savefig(rd / "curves.png", dpi=120); plt.show()

## 5 · Run (each seed is one full run; set `SEEDS` to several for the paper's averaged result)

In [ ]:
results = []
for seed in SEEDS:
    print(f"\n######################## {VARIANT} | seed {seed} ########################")
    results.append(run_experiment(seed))

## 6 · Summary across seeds

In [ ]:
summary = pd.DataFrame(results); pd.set_option("display.width", 200)
print(summary.to_string(index=False))
cols = ["test_acc1", "test_acc5", "test_macro_precision", "test_macro_recall", "test_macro_f1", "test_weighted_f1"]
agg = summary[cols].agg(["mean", "std"]).T
agg["mean ± std"] = agg.apply(lambda r: f"{r['mean']:.2f} ± {0 if np.isnan(r['std']) else r['std']:.2f}", axis=1)
print(f"\n{VARIANT}: results over {len(SEEDS)} run(s)\n", agg[["mean ± std"]].to_string())
summary.to_csv(RUN_DIR / f"summary_{VARIANT}.csv", index=False)
agg.to_csv(RUN_DIR / f"summary_{VARIANT}_mean_std.csv")
print("\nbest model(s):", [f"{r['run_dir']}/best_model.pt" for r in results])